# Воспроизводимые эксперименты RL4TSP

Ноутбук запускает и анализирует самодостаточный набор TSP-экспериментов для проверки end-to-end RL-подходов к задаче коммивояжера. Он устроен как рабочий протокол: сначала задаются параметры запуска, затем создаются конфиги, обучаются веса, после этого отдельно запускаются и анализируются эксперименты.

### Общая цель исследования

Общая цель - проверить три поколения end-to-end RL для TSP в едином воспроизводимом контуре: PointerNet как раннюю sequence-to-sequence pointer-policy, Attention/Transformer policy как более современную attention-based архитектуру и POMO как multi-start развитие policy-gradient подхода. Сравнение проводится не только по средней длине маршрута, но и по свойствам, которые важны для научной интерпретации: перенос на новые размерности, симметрия к переименованию городов, устойчивость к шуму входа, устойчивость к шумной reward-функции и пригодность внутренних сигналов политики как proxy качества.

### Подцели экспериментов

1. Получить сопоставимые checkpoint-модели Attention, PointerNet и POMO, обученные в одном пайплайне.
2. Проверить, как качество и время решения масштабируются от TSP-10 до крупных OOD-размерностей.
3. Проверить, сохраняют ли политики базовую перестановочную симметрию TSP.
4. Проверить робастность маршрутов к шуму координат при оценке.
5. Проверить робастность обучения к шуму в reward-сигнале.
6. Проверить, могут ли entropy, top1-top2 logprob sharpness и sample-length variance служить proxy качества маршрута.

В набор входят пять экспериментальных проверок: масштабирование по числу городов, перестановочная эквивариантность, устойчивость к шуму координат, устойчивость обучения к шумной reward-функции и связь внутренних сигналов политики с качеством маршрута. Подробное обоснование численных гиперпараметров находится в разделе 8, а ограничения статистической интерпретации - в разделе 9.


## 0. Настройка запуска

Эта ячейка задает режим выполнения и каталог результатов. `SCALE = "small"` используется только как smoke-тест: он быстро проверяет, что скрипты импортируются, веса сохраняются, таблицы и графики пишутся в ожидаемые папки. `SCALE = "full"` запускает настройки, предназначенные для интерпретации в работе.

Флаги `RUN_*` позволяют перезапускать только нужные части пайплайна. Это важно при отладке: например, можно один раз обучить веса, затем несколько раз перестраивать сравнение или графики без повторного обучения. Каждый запуск получает уникальный `RUN_ID`, поэтому старые результаты не перезаписываются.

Скрипты запускаются тем же Python-интерпретатором, в котором открыт notebook; ячейка также выставляет `PYTHONPATH=src`, backend Matplotlib и потоковые настройки, поэтому запуск не зависит от имени команды `python3` на Windows.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import Image, display
import sys

PYTHON = sys.executable
PROJECT_ROOT = Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
existing_pythonpath = os.environ.get("PYTHONPATH")
os.environ["PYTHONPATH"] = str(SRC_PATH) if not existing_pythonpath else str(SRC_PATH) + os.pathsep + existing_pythonpath
os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("PYTHONDONTWRITEBYTECODE", "1")
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".runtime-cache" / "matplotlib"))
os.environ.setdefault("RL4TSP_TORCH_THREADS", "4")
SCALE = "small"  # заменить на "full" для итогового запуска
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = PROJECT_ROOT / "results" / RUN_ID
CONFIG_DIR = RUN_ROOT / "configs"

RUN_TRAIN_REINFORCE = True
RUN_TRAIN_POMO = True
RUN_SCALING = True
RUN_DIAGNOSTICS = True
RUN_REWARD_NOISE = True

print(f"Каталог проекта: {PROJECT_ROOT}")
print(f"Режим: {SCALE}")
print(f"Результаты: {RUN_ROOT}")

## 1. Подготовка конфигураций

Эта ячейка создает runtime-конфиги внутри каталога конкретного запуска. Исходные шаблоны лежат в `configs/`, но перед запуском в них подставляются актуальные `output_root` и пути к checkpoint-файлам.

Такой шаг нужен для воспроизводимости: все последующие скрипты читают конфиги из `results/<RUN_ID>/configs/`, поэтому по каталогу результата можно восстановить, какие параметры и какие веса использовались. Сами численные параметры full-запуска обоснованы ниже в разделе 8; здесь меняются только пути текущего запуска.


In [ ]:
def read_config(name: str) -> dict:
    return json.loads((PROJECT_ROOT / "configs" / name).read_text(encoding="utf-8"))


def write_config(name: str, config: dict) -> Path:
    CONFIG_DIR.mkdir(parents=True, exist_ok=True)
    path = CONFIG_DIR / name
    path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
    return path


def reinforce_checkpoint_entries(output_root: Path) -> list[dict]:
    return [
        {"method": "Attention:batch_greedy_mean", "model": "attention", "decoder": "greedy", "checkpoint": str(output_root / "models" / "attention_batch_greedy_mean.pt")},
        {"method": "Attention:greedy_rollout", "model": "attention", "decoder": "greedy", "checkpoint": str(output_root / "models" / "attention_greedy_rollout.pt")},
        {"method": "PointerNet:batch_greedy_mean", "model": "pointer", "decoder": "greedy", "checkpoint": str(output_root / "models" / "pointer_batch_greedy_mean.pt")},
        {"method": "PointerNet:greedy_rollout", "model": "pointer", "decoder": "greedy", "checkpoint": str(output_root / "models" / "pointer_greedy_rollout.pt")},
    ]


def pomo_checkpoint_entry(output_root: Path) -> dict:
    return {"method": "POMO", "model": "attention", "decoder": "pomo", "checkpoint": str(output_root / "models" / "attention_pomo.pt"), "pomo_num_starts": None}


def prepare_configs(scale: str) -> dict[str, Path]:
    if scale not in {"small", "full"}:
        raise ValueError("scale должен быть 'small' или 'full'")
    if RUN_ROOT.exists():
        raise FileExistsError(f"Каталог запуска уже существует: {RUN_ROOT}")
    suffix = "small" if scale == "small" else "full"
    output_root = RUN_ROOT / scale
    paths = {}

    reinforce = read_config(f"tsp_reinforce_{suffix}.json")
    reinforce["output_root"] = str(output_root)
    paths["reinforce"] = write_config("tsp_reinforce.json", reinforce)

    pomo = read_config(f"tsp_pomo_{suffix}.json")
    pomo["output_root"] = str(output_root)
    paths["pomo"] = write_config("tsp_pomo.json", pomo)

    reinforce_entries = reinforce_checkpoint_entries(output_root)
    pomo_entry = pomo_checkpoint_entry(output_root)

    compare = read_config(f"tsp_compare_{suffix}.json")
    compare["output_root"] = str(output_root)
    compare["model_checkpoints"] = reinforce_entries
    compare["pomo_checkpoint"] = str(output_root / "models" / "attention_pomo.pt")
    paths["compare"] = write_config("tsp_compare.json", compare)

    diagnostics = read_config(f"tsp_diagnostics_{suffix}.json")
    diagnostics["output_root"] = str(output_root)
    diagnostics["model_checkpoints"] = reinforce_entries + [pomo_entry]
    diagnostics["entropy_model_checkpoints"] = reinforce_entries + [pomo_entry]
    paths["diagnostics"] = write_config("tsp_diagnostics.json", diagnostics)

    reward_noise = read_config(f"tsp_reward_noise_{suffix}.json")
    reward_noise["output_root"] = str(output_root)
    paths["reward_noise"] = write_config("tsp_reward_noise.json", reward_noise)
    return paths


CONFIG_PATHS = prepare_configs(SCALE)
CONFIG_PATHS

## 2. Обучение весов

Это подготовительный этап для всех последующих сравнений. Обучаются четыре REINFORCE-варианта: Attention и PointerNet с `batch_greedy_mean` и `greedy_rollout` baseline. Отдельно обучается POMO-вариант на Attention-политике с множественными стартами.

Что проверяется на этом этапе: не научная гипотеза сама по себе, а наличие сопоставимых обученных политик для дальнейших экспериментов. Для каждой модели сохраняются веса в `models/` и лог обучения в `logs/`. Лучший checkpoint выбирается по сохраненной метрике обучения, поэтому последующие эксперименты используют не последний случайный шаг, а лучшую найденную версию модели.

Почему именно эти модели и параметры используются, объяснено в разделе 8. Важны два факта: все сравниваемые методы обучаются одним воспроизводимым пайплайном, а последующие эксперименты читают именно эти сохраненные веса.


In [ ]:
def run_step(name: str, *args: str) -> None:
    print(f"\n=== {name} ===", flush=True)
    subprocess.run(args, cwd=PROJECT_ROOT, text=True, check=True, env=os.environ.copy())


if RUN_TRAIN_REINFORCE:
    run_step("REINFORCE: Attention и PointerNet", PYTHON, "experiments/run_tsp_reinforce.py", "--config", str(CONFIG_PATHS["reinforce"]))

if RUN_TRAIN_POMO:
    run_step("POMO", PYTHON, "experiments/run_tsp_pomo.py", "--config", str(CONFIG_PATHS["pomo"]))


## 3. Эксперимент масштабирования

### Гипотеза

Политики, обученные на TSP-20, могут переноситься на другие размеры TSP, но качество должно ухудшаться при сильном уходе от обучающей размерности. Проверяется не превосходство над LKH, а форма деградации качества и вычислительная применимость learned policies.

### Цель

Оценить зависимость optimality gap и времени решения от числа городов для Attention, PointerNet, POMO и опорного решателя. Эксперимент отвечает на главный прикладной вопрос: остается ли RL-политика полезной при росте размерности задачи.

### Фиксированные условия

- Checkpoint-модели берутся из этапа обучения и не дообучаются во время оценки.
- Все методы оцениваются на одних и тех же случайных евклидовых TSP-инстансах внутри каждого размера.
- Координаты генерируются из `[0, 10]^2` с фиксированным seed для воспроизводимости.
- Для `n <= exact_max_n` используется точный Held-Karp; для больших TSP используется LKH как сильный эвристический reference.

### Методика

Для каждого размера TSP генерируется `instances_per_size` независимых инстансов. На каждом инстансе строятся маршруты Attention, PointerNet, POMO и reference solver. В `raw.csv` сохраняются длина маршрута, reference length, gap в процентах и время решения. В `summary.csv` сохраняются среднее, std, sem и 95% доверительный интервал. Графики строятся только из записанного `summary.csv`.

### Интерпретация возможных исходов

Если gap остается умеренным при росте `n_cities`, это показывает переносимость политики за пределы TSP-20. Если gap резко растет, вывод противоположный: модель вычислительно применима, но не сохраняет качество на OOD-размерностях. Если LKH существенно лучше RL, это ожидаемо и не опровергает эксперимент; LKH здесь нужен как сильная точка отсчета, а не как обучаемый baseline.


In [ ]:
if RUN_SCALING:
    run_step("Сравнение качества и времени", PYTHON, "experiments/run_tsp_compare.py", "--config", str(CONFIG_PATHS["compare"]))
    summary_path = RUN_ROOT / SCALE / "tsp_comparison" / "summary.csv"
    run_step("Графики масштабирования", PYTHON, "experiments/plot_tsp_summary.py", "--summary", str(summary_path), "--output-dir", str(RUN_ROOT / SCALE / "figures"))

scaling_summary = pd.read_csv(RUN_ROOT / SCALE / "tsp_comparison" / "summary.csv")
display(scaling_summary)
for image_path in [RUN_ROOT / SCALE / "figures" / "tsp_gap.png", RUN_ROOT / SCALE / "figures" / "tsp_time.png"]:
    display(Image(filename=str(image_path)))

## 4. Перестановочная эквивариантность

### Гипотеза

Корректная TSP-политика не должна зависеть от произвольной нумерации городов. Если входные точки переставить, а затем перевести полученный маршрут обратно в исходную нумерацию, канонический тур должен совпадать или почти совпадать с исходным.

### Цель

Проверить базовое структурное свойство learned policies: чувствительны ли Attention, PointerNet и POMO к порядку подачи городов. Это диагностический тест корректности поведения, а не основной тест качества маршрута.

### Фиксированные условия

- Используются обученные checkpoint-модели без дообучения.
- Размер TSP фиксирован на обучающей размерности, чтобы не смешивать эквивариантность с OOD-переносом.
- Для каждого базового инстанса генерируется фиксированное число случайных перестановок.
- Сравнение проводится после обратного отображения маршрута в исходную нумерацию и канонизации тура.

### Методика

Для каждого исходного TSP-инстанса модель строит базовый маршрут. Затем координаты городов переупорядочиваются случайной перестановкой, модель строит новый маршрут, а индексы переводятся обратно. После канонизации сравниваются два тура. В таблицах сохраняются `hamming_distance`, абсолютная разница длины и `perfect_match`.

### Интерпретация возможных исходов

Если `hamming_mean` близок к нулю и `perfect_match_rate` высок, политика практически сохраняет перестановочную симметрию. Если `hamming_mean` велик, результат означает зависимость от входного порядка. Такой исход не обязательно сразу делает модель бесполезной, но ослабляет научную интерпретацию learned policy как геометрического TSP-решателя.


In [ ]:
if RUN_DIAGNOSTICS:
    run_step("Диагностики TSP", PYTHON, "experiments/run_tsp_diagnostics.py", "--config", str(CONFIG_PATHS["diagnostics"]))

permutation_summary = pd.read_csv(RUN_ROOT / SCALE / "tsp_diagnostics" / "permutation_invariance_summary.csv")
display(permutation_summary)
display(Image(filename=str(RUN_ROOT / SCALE / "figures" / "permutation_invariance.png")))

## 5. Устойчивость к шуму координат

### Гипотеза

Если политика действительно использует геометрию TSP устойчивым образом, малый шум координат не должен приводить к резкому ухудшению маршрута, измеренного на истинных координатах. Сильный шум должен ухудшать качество постепенно, а не хаотически.

### Цель

Измерить робастность обученных политик к ошибке во входных координатах. Это отдельная проверка от reward-noise: здесь меняется вход при построении маршрута, но сама модель уже обучена обычным образом.

### Фиксированные условия

- Базовые TSP-инстансы генерируются один раз и используются для всех уровней шума.
- Reference length считается на истинных координатах.
- Модель получает зашумленные координаты, но итоговая длина маршрута оценивается на истинных координатах.
- Уровни `sigma` заданы как абсолютный шум при масштабе координат `scale = 10.0`.

### Методика

Для каждого базового инстанса сначала считается reference solution. Затем к координатам добавляется гауссов шум, координаты обрезаются в допустимый диапазон, модель строит маршрут по зашумленному входу, а длина маршрута пересчитывается на исходных координатах. Для каждого уровня шума строится средний gap и 95% доверительный интервал.

### Интерпретация возможных исходов

Плоская кривая gap означает устойчивость к малым ошибкам координат. Быстрый рост gap означает высокую чувствительность к геометрическому шуму. Если методы различаются по наклону кривой, это можно трактовать как различие в робастности, но строгие попарные утверждения требуют парного теста по инстансам.


In [ ]:
noise_summary = pd.read_csv(RUN_ROOT / SCALE / "tsp_diagnostics" / "noise_robustness_summary.csv")
display(noise_summary)
display(Image(filename=str(RUN_ROOT / SCALE / "figures" / "noise_robustness.png")))



## 6. Reward-noise

### Гипотеза

Policy-gradient обучение должно ухудшаться при росте шума в наблюдаемой reward-функции, потому что advantage становится менее надежной оценкой направления улучшения политики. Более устойчивый метод должен показывать меньшую деградацию истинного качества при одинаковом уровне reward-noise.

### Цель

Проверить, насколько Attention, PointerNet и POMO чувствительны к ошибке в обучающем сигнале. Эксперимент моделирует ситуацию, где во время обучения доступна не точная длина тура, а зашумленная оценка.

### Фиксированные условия

- Обучение остается на TSP-20, как в основной постановке.
- Для каждого `sigma_rel` модель обучается заново и сохраняет checkpoint.
- Evaluation reference set строится один раз и переиспользуется для всех методов и уровней шума.
- Оценка после обучения всегда проводится по истинной длине маршрута, а не по зашумленной reward.

### Методика

Во время обучения к длине тура добавляется относительный шум. Advantage считается по наблюдаемой зашумленной длине, но в логах также сохраняется истинная длина. После обучения модель оценивается на фиксированном наборе TSP-инстансов. В `raw.csv` сохраняются обучающие траектории и eval-строки; в `summary.csv` сохраняются агрегированные значения и 95% доверительные интервалы для eval-групп.

### Интерпретация возможных исходов

Если `sigma_rel = 0.0` заметно лучше шумных вариантов, обучение чувствительно к ошибке reward. Если качество стабильно при умеренном шуме, метод устойчив к неточному обучающему сигналу. Если различия между соседними уровнями малы, нельзя делать сильное утверждение без дополнительных training seeds; текущий CI отражает eval-инстансы, а не межseedовую дисперсию обучения.


In [ ]:
if RUN_REWARD_NOISE:
    run_step("Reward-noise", PYTHON, "experiments/run_tsp_reward_noise.py", "--config", str(CONFIG_PATHS["reward_noise"]))

reward_summary = pd.read_csv(RUN_ROOT / SCALE / "tsp_reward_noise" / "summary.csv")
display(reward_summary)
for image_path in [RUN_ROOT / SCALE / "figures" / "reward_noise_gap.png", RUN_ROOT / SCALE / "figures" / "reward_noise_training.png"]:
    display(Image(filename=str(image_path)))

## 7. Энтропия политики и качество решения

### Гипотеза

Внутренние сигналы политики могут быть proxy качества маршрута только если они устойчиво связаны с optimality gap. Проверяются три кандидата: средняя энтропия действий, top1-top2 logprob sharpness и дисперсия длины при повторном стохастическом декодировании.

### Цель

Проверить, можно ли без запуска reference solver оценивать риск плохого маршрута по внутренним сигналам learned policy. Это наиболее самостоятельный исследовательский блок: отрицательный результат здесь тоже содержателен, потому что показывает ограниченность простой self-assessment диагностики.

### Фиксированные условия

- Используются те же checkpoint-модели, что и в остальных экспериментах.
- Для каждого размера TSP генерируются независимые eval-инстансы.
- Gap считается относительно Held-Karp или LKH reference в зависимости от размера.
- Число стохастических сэмплов фиксировано параметром `entropy_stochastic_samples`.

### Методика

На каждом инстансе модель строит greedy маршрут и сохраняет gap. Из распределений действий извлекаются entropy и logprob sharpness; дополнительно несколько раз запускается stochastic decode и считается variance длины маршрута. Для каждого сигнала считается Pearson r с gap как тест линейной связи и Spearman rho как тест монотонной связи. Для обеих корреляций сохраняются p-value и приближенный 95% доверительный интервал.

### Интерпретация возможных исходов

Если Pearson и Spearman показывают сильную связь с gap, сигнал можно рассматривать как грубый proxy качества и использовать для triage инстансов без reference solver. Если обе корреляции слабы для всех трех сигналов, корректный вывод отрицательный: простая уверенность или вариативность политики не является надежной самооценкой качества маршрута на проверенных размерах. Если Pearson слабый, а Spearman сильный, связь скорее монотонная, но нелинейная; тогда график scatter должен использоваться для уточнения формы зависимости.


In [ ]:
entropy_summary = pd.read_csv(RUN_ROOT / SCALE / "tsp_diagnostics" / "entropy_experiment_summary.csv")
display(entropy_summary)
for image_path in [RUN_ROOT / SCALE / "figures" / "entropy_gap.png", RUN_ROOT / SCALE / "figures" / "entropy_correlations.png", RUN_ROOT / SCALE / "figures" / "entropy_spearman_correlations.png"]:
    display(Image(filename=str(image_path)))

## 8. Обоснование выбора гиперпараметров full-запуска

Малые конфиги `*_small.json` не являются экспериментальными настройками. Они нужны только для smoke-проверки: проверить импорт модулей, создание весов, запись CSV/PNG и корректность путей. Научная интерпретация относится к full-конфигам.

### Общие параметры

- `seed = 42/43/44/46`: фиксирует генераторы случайных чисел и делает запуск воспроизводимым. Разные seed по блокам отделяют случайные потоки обучения, POMO, масштабирования и диагностик, чтобы один и тот же набор TSP-инстансов не использовался во всех ролях одновременно.
- `scale = 10.0`: координаты городов сэмплируются из `[0, 10]^2`. Постоянное масштабирование не меняет оптимальный маршрут, но дает длины маршрутов порядка десятков, что удобно для численной стабильности REINFORCE и чтения таблиц.
- `device = cpu`: выбран как базовый воспроизводимый режим для ревьюера. Код не требует GPU, поэтому результат можно проверить на обычной машине.
- `progress = true`: включает tqdm-прогресс, чтобы длительные full-запуски не выглядели зависшими.
- `output_root`, `experiment_name`: задают только структуру артефактов и не влияют на результаты. Каждый запуск пишет в новый каталог `results/<run_id>/...`.

### Архитектура и обучение REINFORCE/POMO

- `models = [attention, pointer]`: сравниваются две разные нейросетевые политики для TSP: attention-based policy и PointerNet. POMO обучается отдельно как attention-policy с множественными стартами.
- `hidden_dim = 128`: единая размерность скрытого состояния для Attention и PointerNet. Это достаточно большая модель для TSP-20, но она остается легкой для CPU-запуска. Для Attention 128 делится на 8 голов внимания без остатка.
- `Attention num_layers = 3`: фиксировано в коде для Attention-политики. Три encoder-слоя дают нетривиальное взаимодействие между городами, но не делают модель слишком тяжелой для повторного запуска.
- `n_train = 20`: обучение ведется на TSP-20. Это намеренно: масштабирование затем проверяет перенос на `10, 20, 50, 100, 200, 300` городов, включая OOD-размерности.
- `epochs = 3000`: полный запуск дает заметную динамику обучения и сохраняет практичное время воспроизведения. Лучшие веса сохраняются отдельно, поэтому поздняя деградация политики не затирает лучший checkpoint.
- `learning_rate = 0.0001`: консервативный шаг Adam для стохастических policy-gradient обновлений. Он выбран для устойчивости, а не для агрессивного достижения лучшего результата за минимальное число эпох.
- `gradient clipping = 1.0`: задано в коде обучения. Ограничивает редкие большие градиенты REINFORCE/POMO и снижает риск нестабильных скачков loss.
- `baselines = [batch_greedy_mean, greedy_rollout]`: оставляет исходный baseline для сопоставимости и добавляет более корректный per-instance greedy rollout baseline.
- `batch_size = 64` для REINFORCE: снижает дисперсию градиента при умеренной стоимости эпохи на CPU.
- `POMO batch_size = 8`, `num_starts = 20`: POMO разворачивает каждый TSP-20 инстанс в 20 фиксированных стартов, поэтому эффективный батч равен 160 rollout. Больший физический batch резко увеличил бы время и память.
- `eval_interval = 50` для POMO: greedy POMO-оценка дороже обычного rollout, поэтому она считается не каждую эпоху, а регулярно. Это ускоряет обучение без изменения самой обучающей цели.
- `pomo_num_starts = null` в оценке: означает использовать все доступные старты для размера задачи. Для TSP-20 это 20 стартов; для других размеров используется полный набор городов как стартов, если это поле не переопределено.

### Масштабирование и опорные решения

- `sizes = [10, 20, 50, 100, 200, 300]`: включает область меньше обучения, обучающий размер TSP-20 и крупные OOD-размерности. Это прямо отвечает на вопрос о переносе политики при росте задачи.
- `instances_per_size = 100`: дает 100 независимых инстансов на каждый размер. Этого достаточно для 95% доверительных интервалов, но LKH-часть остается вычислительно приемлемой.
- `exact_max_n = 10`: Held-Karp имеет экспоненциальную сложность `O(n^2 2^n)`, поэтому точное решение используется только на малых задачах.
- `reference_solver = exact_or_lkh`: для `n <= 10` используется точное DP-решение, для больших размерностей LKH как сильный эвристический референс. Это лучше, чем сравнение только с точным решателем, который не масштабируется.
- `lkh_scale = 1000000`: LKH работает с целочисленными расстояниями; множитель `10^6` сохраняет шесть знаков после запятой и почти не искажает евклидову геометрию.

### Диагностики: перестановки, шум координат, self-assessment signals

- `permutation_n_cities = 20`: проверка перестановочной эквивариантности проводится на обучающей размерности, где модель должна быть наиболее стабильной.
- `permutation_instances = 100`, `permutations_per_instance = 5`: всего 500 парных сравнений. Это достаточно для доверительного интервала и не превращает проверку в основной расчет.
- `noise_n_cities = 20`: устойчивость к шуму координат проверяется на обучающей размерности, чтобы измерять именно чувствительность политики, а не смешивать ее с OOD-переносом.
- `noise_sigmas = [0.0, 0.1, 0.2, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]`: сетка идет от контроля без шума до сильного геометрического искажения. При `scale = 10` это соответствует 0-50% масштаба координат.
- `noise_instances = 500`: больше, чем в масштабировании, потому что сравниваются уровни шума, а ожидаемые различия могут быть небольшими.
- `entropy_sizes = [20, 30, 50, 100]`: TSP-20 проверяет in-distribution, TSP-30/50/100 проверяют перенос сигналов самодиагностики на OOD-размерности.
- `entropy_instances = 300`: нужно для устойчивой оценки Pearson r и Spearman rho между качеством маршрута и внутренними сигналами политики.
- `entropy_stochastic_samples = 8`: восемь стохастических декодирований дают оценку дисперсии длины маршрута. Большее число сэмплов сильно удорожает эксперимент, но не меняет основной вопрос: являются ли entropy, sharpness и sample variance хорошими предикторами качества.
- `run_permutation/run_noise/run_entropy = true`: все три диагностических блока включены в full-запуск, потому что они отвечают на разные части критики: симметрии, устойчивость и интерпретируемость внутренних сигналов политики.

### Reward-noise experiment

- `models = [attention, pointer, pomo]`: reward-noise проверяется для всех основных обучаемых подходов, чтобы вывод не зависел от одной архитектуры.
- `baselines = [greedy_rollout]`: в этом эксперименте используется основной корректный baseline; добавление `batch_greedy_mean` удвоило бы стоимость без изменения проверяемой гипотезы о шуме награды.
- `train_size = 20`: обучение остается на TSP-20, чтобы reward-noise сравнивался с основной постановкой REINFORCE/POMO.
- `n_train = 64`: число обучающих инстансов на эпоху согласовано с REINFORCE batch size и дает стабильную оценку noisy advantage.
- `eval_sizes = [10, 20, 50, 100, 200]`: оценка включает in-distribution и OOD-размерности, но исключает 300, чтобы reward-noise full-запуск не становился чрезмерно дорогим.
- `eval_instances = 100`: одинаковая статистическая база с масштабированием; достаточно для 95% доверительных интервалов.
- `sigma_rel_values = [0.0, 0.01, 0.05, 0.1, 0.2]`: включает контроль без шума, слабый шум, умеренный шум и заметно искаженную награду. Диапазон проверяет, насколько быстро ухудшается обучение при росте ошибки в reward.



## 9. Статистическая значимость: обоснование и ограничения

Во всех full-запусках статистика считается по независимо сгенерированным TSP-инстансам. На графиках с агрегированными средними точки или столбцы показывают выборочное среднее, а интервалы показывают 95% доверительный интервал для среднего: `t_{0.975, n-1} * std / sqrt(n)`. Для корреляций в entropy-эксперименте используются Pearson r и Spearman rho; 95% интервалы считаются через Fisher z-преобразование как приближенная оценка неопределенности корреляции. Эти интервалы показывают неопределенность оценки на выбранной выборке, но не являются автоматическим доказательством превосходства метода.


### Где используются std, CI и заливка

- В `summary.csv` сохраняются и `*_std`, и `*_sem`, и `*_ci95`. `*_std` - это выборочное стандартное отклонение наблюдений внутри группы. Оно нужно для аудита разброса, но не является доверительным интервалом среднего.
- На основных графиках с агрегированными средними используются `*_ci95` как вертикальные error bars. Если по технической причине `*_ci95` отсутствует, код может fallback-нуть к `*_std`, но в full-запусках ожидаются именно CI-колонки.
- Заливка вокруг линии оставлена только на графике `noise_robustness.png`, потому что там надо визуально показать непрерывную деградацию качества по уровню шума координат. В `tsp_gap.png`, `tsp_time.png`, `reward_noise_gap.png` и `entropy_correlations.png` используются только вертикальные CI-интервалы без заливки.
- `entropy_correlations.png` показывает Pearson r и 95% CI, а `entropy_spearman_correlations.png` показывает Spearman rho и 95% CI. Это не std маршрутов и не std сигналов политики.
- `entropy_gap.png` является scatter plot по отдельным инстансам; там нет std/CI, потому что задача графика - показать облако точек и форму связи.
- `reward_noise_training.png` показывает обучающие траектории одного запуска/seed. Там нет CI и std: это диагностический график динамики обучения, а не статистическая оценка среднего по независимым training seeds.
- В entropy-таблицах поле `sample_length_std` означает стандартное отклонение длины маршрута при нескольких стохастических декодированиях одного и того же инстанса. Это self-assessment signal модели, а не доверительный интервал по датасету.

### Общие ограничения интерпретации

- Доверительные интервалы относятся к распределению случайных евклидовых TSP-инстансов, заданному генератором, а не ко всем возможным TSP-задачам.
- Для `n_cities <= exact_max_n = 10` отклонение считается от точного Held-Karp решения. Для больших размерностей отклонение считается от LKH, то есть от сильного эвристического референса, а не от доказанного оптимума.
- Методы сравниваются на одних и тех же инстансах внутри каждого размера TSP, поэтому строгая проверка различий между двумя методами должна быть парной: paired bootstrap, paired t-test по per-instance gap или Wilcoxon. Текущие графики показывают marginal 95% CI для каждого метода; пересечение/непересечение интервалов является только визуальной эвристикой.
- В экспериментах много методов, размеров и уровней шума. Формальная серия утверждений вида «метод A статистически лучше B во всех точках» потребовала бы поправки на множественные сравнения или заранее выбранного основного сравнения. В текущей работе корректнее говорить о трендах, устойчивости и величине эффекта.
- Smoke-конфиги не дают статистической значимости. Они проверяют только, что код запускается, пишет файлы и строит графики.

### 1. Масштабирование

- Full-конфиг использует `sizes = [10, 20, 50, 100, 200, 300]` и `instances_per_size = 100`. Для каждого размера строится отдельная выборка из 100 TSP-инстансов, поэтому CI оценивает среднее качество/время именно для этого размера.
- Для TSP-10 референс точный, поэтому gap имеет строгий смысл отклонения от оптимума. Для TSP-20 и выше gap является отклонением от LKH; это корректно как практическое сравнение с сильной эвристикой, но не как доказательство оптимальности.
- Для TSP-100/200/300 100 инстансов достаточно для грубой оценки среднего тренда и доверительного интервала. Однако при малой разнице между методами или широкой дисперсии gap нельзя уверенно утверждать превосходство только по графику.
- Если сетку расширять до TSP-400 при том же `instances_per_size = 100`, такой результат надо называть exploratory. Размер задачи растет, дисперсия качества и времени обычно растет, а LKH-оценка становится дороже; для сильного утверждения по TSP-400 лучше увеличить число инстансов до 200-300 или явно ограничить вывод словами «наблюдаемый тренд».
- Важно не смешивать `n_cities` и статистический размер выборки. TSP-300 не становится статистически надежнее из-за 300 городов; надежность задается числом независимых инстансов, здесь `n = 100`.

### 2. Перестановочная эквивариантность

- Используются `permutation_instances = 100` и `permutations_per_instance = 5`, то есть 500 сравнений на метод. Это хорошо выявляет грубые нарушения симметрии.
- Ограничение: пять перестановок одного исходного инстанса не являются полностью независимыми наблюдениями. Формальный effective sample size ближе к 100 исходным инстансам, чем к 500 строкам.
- Поэтому результат следует трактовать как диагностическую проверку архитектурного свойства, а не как строгий hypothesis test. Для строгого интервала лучше использовать cluster bootstrap по исходным инстансам.

### 3. Устойчивость к шуму координат

- Используются `noise_instances = 500` и сетка `noise_sigmas`. Для каждого уровня шума среднее и CI считаются по 500 базовым TSP-инстансам, что дает более устойчивую оценку, чем в масштабировании.
- Одни и те же базовые инстансы проходят через разные уровни шума, поэтому сравнение между соседними `sigma` фактически парное. Текущие CI показывают неопределенность среднего на каждом уровне, но не проверяют строго разность между уровнями.
- Вывод должен формулироваться как форма кривой деградации: растет ли средний gap и насколько быстро. Для утверждения «sigma A хуже sigma B» лучше использовать paired bootstrap по инстансам.

### 4. Энтропия политики и качество решения

- Используются `entropy_instances = 300` на каждый размер из `entropy_sizes = [20, 30, 50, 100]`. Это достаточно для устойчивой оценки Pearson r и Spearman rho между gap и тремя сигналами: entropy, top1-top2 logprob sharpness, sample-length variance.
- Здесь основная статистика не средний gap, а сила связи. Pearson r проверяет линейную связь, Spearman rho проверяет монотонную связь; оба графика показывают 95% CI для корреляции, а не std маршрутов.
- Если все три сигнала имеют слабую связь с gap и интервалы остаются далеко от сильной корреляции, корректный вывод: эти self-assessment signals являются слабыми предикторами качества на проверенных размерах.
- Ограничение: Pearson r проверяет линейную связь, Spearman rho - только монотонную. Если оба показателя малы, это не исключает сложную нелинейную зависимость, но исключает простую линейную или монотонную proxy-интерпретацию проверенных сигналов.

### 5. Reward-noise

- Full-конфиг использует `sigma_rel_values = [0.0, 0.01, 0.05, 0.1, 0.2]`, `eval_instances = 100` и `eval_sizes = [10, 20, 50, 100, 200]`. Для каждой пары `(sigma, size)` оценка строится по 100 инстансам.
- `sigma = 0.0` является контрольной группой без шума. Остальные значения показывают, как обучение деградирует при ошибке в reward.
- При 100 eval-инстансах можно уверенно обсуждать крупные эффекты и общую монотонность ухудшения. Небольшие различия между соседними sigma или между методами требуют парного теста или большего числа eval-инстансов.
- Обучение само по себе стохастично, но full-конфиг фиксирует один seed на вариант. Поэтому CI в reward-noise отражают разброс eval-инстансов, а не разброс между независимыми training seeds. Для строгого вывода о стабильности обучения нужны 3-5 независимых training seeds на каждый вариант, что существенно дороже.

### Формулировка для текста работы

Корректная формулировка: «На графиках приведены средние значения и 95% доверительные интервалы по независимым TSP-инстансам. Интервалы используются для оценки устойчивости наблюдаемых трендов. Строгие попарные утверждения о превосходстве методов не делаются без отдельного парного теста; для больших размерностей результаты относительно LKH интерпретируются как сравнение с сильным эвристическим референсом, а не как отклонение от доказанного оптимума».




## 10. Краткая интерпретация для защиты

Сильная часть работы: построен воспроизводимый набор экспериментов RL4TSP, где Attention, PointerNet и POMO сравниваются на одних и тех же задачах, с едиными опорными решениями, доверительными интервалами и сохраненными весами моделей.

Исследовательская часть: эксперимент с сигналами самодиагностики проверяет, можно ли использовать энтропию, sharpness и дисперсию сэмплов как индикаторы качества решения. Если Pearson/Spearman корреляции стабильно слабые, это самостоятельный отрицательный результат: политика может быть уверенной или неуверенной, но эти внутренние сигналы не дают надежной оценки оптимальности маршрута на новых размерах задачи.

## 11. Итоговая критическая оценка проекта

### Что проект научно проверяет

Проект следует одной исследовательской линии: обучить компактные RL-политики для евклидовой TSP-20, затем проверить не только среднее качество маршрутов, но и свойства, которые важны для доверия к методу. Поэтому эксперименты идут от базового сравнения качества и времени к диагностике симметрии, устойчивости к ошибке входа, устойчивости к шумной награде и проверке внутренних сигналов самооценки политики.

Корректная область утверждений узкая и достаточная: результаты относятся к случайным евклидовым TSP-инстансам, к реализованным вариантам Attention, PointerNet и POMO, к REINFORCE/POMO-обучению на TSP-20 и к заданным full-конфигам. Код не доказывает, что RL превосходит LKH. Он проверяет, как обученные политики ведут себя относительно сильного опорного решателя, насколько они переносятся на новые размерности и дают ли их внутренние вероятностные сигналы полезную самооценку качества.

### Что является сильной частью работы

Сильная часть - воспроизводимый экспериментальный контур. Все скрипты читают явные JSON-конфиги, пишут `raw.csv`, `summary.csv`, checkpoint-файлы и PNG в фиксированную структуру `results/<RUN_ID>/<scale>/...`. Старые запуски не перезаписываются без явного разрешения. Графики строятся из уже записанных CSV, поэтому их можно пересоздать и проверить отдельно от обучения.

Вторая сильная часть - сравнение не ограничено одной моделью. Attention, PointerNet и POMO проходят через одни и те же evaluation инстансы, а для больших TSP используется LKH как практический референс. Это закрывает главный методический риск: нельзя интерпретировать качество RL-модели без сильной внешней точки сравнения.

Третья сильная часть - entropy-блок. Он проверяет не один сигнал, а три: энтропию, top1-top2 logprob sharpness и дисперсию длины при стохастическом декодировании. Если все три дают слабую Pearson/Spearman связь с gap, это не провал, а корректный отрицательный результат: простые self-assessment signals не являются надежной заменой опорной оценки качества на OOD TSP.

### Границы корректной интерпретации

Названия Attention, PointerNet и POMO обозначают реализованные семейства подходов, а не буквальную репродукцию всех деталей исходных статей. Attention-модель здесь является Transformer-encoder pointer policy; PointerNet является LSTM pointer policy; POMO реализован как multi-start group-baseline обучение и best-of-starts inference. Это надо формулировать именно так, без утверждения о полном воспроизведении оригинальных реализаций.

Gap для TSP-10 считается относительно точного Held-Karp решения. Для больших размеров gap считается относительно LKH. Это корректно для практического сравнения, но не является отклонением от доказанного оптимума. Доверительные интервалы на графиках являются marginal 95% CI по инстансам; они помогают оценить устойчивость тренда, но не заменяют парный тест между методами и не учитывают множественные сравнения.

Best checkpoint выбирается по метрике обучения, а не по отдельному validation set. Это снижает риск случайно взять поздний ухудшившийся шаг, но не является строгим validation model selection. Reward-noise использует один training seed на вариант, поэтому CI там описывают разброс eval-инстансов, а не межseedовую устойчивость обучения.

### Согласованная итоговая формулировка

Проект следует защищать так: «Я построил воспроизводимый набор экспериментов RL4TSP, где несколько нейросетевых политик сравниваются с точным/LKH референсом и проходят диагностические проверки переносимости, симметрии, робастности и самооценки. Основной вывод не в том, что RL побеждает LKH, а в том, какие свойства такие политики реально демонстрируют и где их ограничения. Наиболее самостоятельный исследовательский результат - проверка entropy/sharpness/sample-variance сигналов по Pearson и Spearman: если их корреляция с качеством слаба, значит простая уверенность политики не дает надежного критерия качества маршрута на новых размерах TSP».
